In [ ]:
!pip -q install beautifulsoup4 lxml requests

In [ ]:
import pathlib, json, requests
from bs4 import BeautifulSoup
from google.colab import files

URL = "https://guutit.com/resources/Demo-Files/Hypostat-Final-V6.html"
OUT_HTML = "hypostat_click_help_toggle_sticky.html"

# --- fetch & parse ---
html = requests.get(URL, timeout=30).text
soup = BeautifulSoup(html, "lxml")

# --- CSS ---
style = soup.new_tag("style")
style.string = """
:root{--ha-bg:#0b1220;--ha-fg:#e6edf3;--ha-b:#243042;--ha-ac:#60a5fa}
#ha-toggle{position:fixed;right:18px;bottom:18px;z-index:2147483645;width:48px;height:48px;border-radius:50%;
  border:1px solid var(--ha-b);background:var(--ha-bg);color:var(--ha-fg);display:grid;place-items:center;
  box-shadow:0 10px 30px rgba(0,0,0,.35);cursor:pointer;font-weight:600;font-size:16px;line-height:1}
#ha-toggle.ha-armed{outline:2px solid var(--ha-ac);box-shadow:0 0 20px rgba(96,165,250,.6),0 10px 30px rgba(0,0,0,.6)}
#ha-mask{display:none;position:fixed;inset:0;background:rgba(0,0,0,.55);z-index:2147483638}
#ha-spot{display:none;position:absolute;z-index:2147483639;pointer-events:none;border-radius:14px;outline:2px solid var(--ha-ac);
  box-shadow:0 0 0 200vmax rgba(0,0,0,.45)}
#ha-tip{display:none;position:fixed;z-index:2147483640;max-width:360px;background:#121a2a;color:var(--ha-fg);
  border:1px solid var(--ha-b);border-radius:12px;padding:12px;box-shadow:0 20px 60px rgba(0,0,0,.45)}
#ha-tip h4{margin:0 0 6px 0;font-size:14px}
#ha-tip .bodycopy{font-size:13px;line-height:1.35}
#ha-tip .row{display:flex;gap:6px;justify-content:flex-end;margin-top:8px}
.ha-btn{border:1px solid var(--ha-b);background:#121a2a;color:var(--ha-fg);border-radius:10px;padding:6px 10px;font-size:12px;cursor:pointer}
"""
soup.head.append(style)

# --- overlay nodes ---
soup.body.append(BeautifulSoup('<button id="ha-toggle" title="Click to enter help mode">?</button>', "lxml"))
soup.body.append(BeautifulSoup('<div id="ha-mask"></div>', "lxml"))
soup.body.append(BeautifulSoup('<div id="ha-spot"></div>', "lxml"))
soup.body.append(BeautifulSoup('''
<div id="ha-tip">
  <h4 id="ha-tt">Title</h4>
  <div id="ha-tb" class="bodycopy"></div>
  <div class="row"><button class="ha-btn" id="ha-close">Close</button></div>
</div>''', "lxml"))

# --- guidance steps (same regions) ---
steps = [
  {"type":"percent","base":"#sdcanvasHolder,#sdcanvas","title":"Top Navigation","body":"Use tabs to switch major views.","left":0.00,"top":0.00,"width":1.00,"height":0.025},
  {"type":"percent","base":"#sdcanvasHolder,#sdcanvas","title":"Filters Panel","body":"Overall filter panel – pick metric, country, year.","left":0.00,"top":0.03,"width":1.01,"height":0.165},
  {"type":"percent","base":"#sdcanvasHolder,#sdcanvas","title":"Metric Selector","body":"Choose which indicator to analyze.","left":0.00,"top":0.06,"width":0.40,"height":0.13},
  {"type":"percent","base":"#sdcanvasHolder,#sdcanvas","title":"Country Filter","body":"Select countries (All or individual).","left":0.44,"top":0.06,"width":0.12,"height":0.12},
  {"type":"percent","base":"#sdcanvasHolder,#sdcanvas","title":"Year Filter","body":"Select years (All or a range).","left":0.67,"top":0.06,"width":0.12,"height":0.12},
  {"type":"percent","base":"#sdcanvasHolder,#sdcanvas","title":"Reset Filters","body":"Click to clear filters to defaults.","left":0.87,"top":0.05,"width":0.13,"height":0.04},
  {"type":"percent","base":"#sdcanvasHolder,#sdcanvas","title":"Chart Title","body":"Context for the current view.","left":0.00,"top":0.20,"width":0.70,"height":0.04},
  {"type":"percent","base":"#sdcanvasHolder,#sdcanvas","title":"Series Legend","body":"Click a legend item to toggle a country/series.","left":0.00,"top":0.26,"width":0.96,"height":0.075},
  {"type":"percent","base":"#sdcanvasHolder,#sdcanvas","title":"Main Visualization","body":"Primary plot area – hover for tooltips.","left":0.00,"top":0.34,"width":1.01,"height":0.22},
  {"type":"percent","base":"#sdcanvasHolder,#sdcanvas","title":"Status / Tabs","body":"Bottom controls, summaries, or pagination.","left":0.00,"top":0.58,"width":1.01,"height":0.44},
]
cfg_tag = soup.new_tag("script")
cfg_tag.string = "window.__HELPER_AGENT_CONFIG__ = " + json.dumps({"steps": steps}, ensure_ascii=False) + ";"
soup.body.append(cfg_tag)

# --- JS (sticky help mode, multi-click/touch) ---
runtime = soup.new_tag("script")
runtime.string = r"""
(()=>{const $=(s,r=document)=>r.querySelector(s), $$=(s,r=document)=>Array.from(r.querySelectorAll(s));
const cfg=window.__HELPER_AGENT_CONFIG__||{steps:[]};

const btnToggle=$('#ha-toggle'), mask=$('#ha-mask'), spot=$('#ha-spot'),
      tip=$('#ha-tip'), tt=$('#ha-tt'), tb=$('#ha-tb'), btnClose=$('#ha-close');

let helpArmed=false, currentIdx=-1, currentBase=null;

// --- geometry helpers ---
function pickBase(selCSV){
  const tries=(selCSV||"").split(",").map(s=>s.trim()).filter(Boolean);
  for(const s of tries){ const el=$(s); if(el){ const r=el.getBoundingClientRect(); if(r.width>40&&r.height>40) return el; } }
  let best=null,area=0; for(const c of $$("canvas")){ const r=c.getBoundingClientRect(); const a=Math.max(0,r.width)*Math.max(0,r.height); if(a>area){best=c;area=a;} }
  return best||document.body;
}
function percentRectViewport(step){
  const base=pickBase(step.base); currentBase=base;
  const rb=base.getBoundingClientRect();
  const L=step.left??0, T=step.top??0, W=step.width??1, H=step.height??1;
  return { left:rb.left+rb.width*L, top:rb.top+rb.height*T,
           width:Math.max(1,rb.width*W), height:Math.max(1,rb.height*H), rb, base };
}
function toContainerCoords(r, base){
  const rb=r.rb||base.getBoundingClientRect();
  const left=(r.left-rb.left)+base.scrollLeft, top=(r.top-rb.top)+base.scrollTop;
  return {left, top, width:r.width, height:r.height};
}
function placeTip(rV){
  const b=tip.getBoundingClientRect(), above=rV.top-b.height-10>0;
  tip.style.left=Math.min(Math.max(12, rV.left+rV.width/2-b.width/2), window.innerWidth-b.width-12)+'px';
  tip.style.top=(above? rV.top-b.height-10 : rV.top+rV.height+10)+'px';
}
function ensureSpotParent(base){
  if(spot.parentElement!==base){ try{ base.appendChild(spot); }catch{} }
  spot.style.position='absolute'; spot.style.pointerEvents='none';
}

// --- draw/hide ---
function showStep(idx){
  const s=cfg.steps[idx]; if(!s) return;
  const rV=percentRectViewport(s), base=rV.base||document.body;
  ensureSpotParent(base);
  const rC=toContainerCoords(rV, base), pad=8;

  mask.style.display='block'; tip.style.display='block'; spot.style.display='block';
  spot.style.left=Math.max(0,rC.left-pad)+'px';
  spot.style.top=Math.max(0,rC.top-pad)+'px';
  spot.style.width=Math.max(8,rC.width+pad*2)+'px';
  spot.style.height=Math.max(8,rC.height+pad*2)+'px';

  tt.textContent=s.title||'Info';
  tb.textContent=s.body||'';
  placeTip(rV);
  currentIdx=idx;
}
// hides current overlay ONLY (keeps helpArmed as-is)
function hideOverlay(){
  currentIdx=-1;
  spot.style.display='none'; mask.style.display='none'; tip.style.display='none';
}
function arm(){ helpArmed=true; btnToggle.classList.add('ha-armed'); btnToggle.title='Help mode ON. Tap/click areas to learn more. Click ? to exit.'; }
function disarm(){ helpArmed=false; btnToggle.classList.remove('ha-armed'); btnToggle.title='Click to enter help mode'; hideOverlay(); }

// --- event utils (click + touch) ---
function getClientXY(e){
  if(e.touches&&e.touches.length){ return {x:e.touches[0].clientX, y:e.touches[0].clientY}; }
  if(e.changedTouches&&e.changedTouches.length){ return {x:e.changedTouches[0].clientX, y:e.changedTouches[0].clientY}; }
  return {x:e.clientX, y:e.clientY};
}
function pointInRect(px,py,r){ return px>=r.left && py>=r.top && px<=r.left+r.width && py<=r.top+r.height; }
function stepAtPoint(x,y){
  for(let k=cfg.steps.length-1;k>=0;k--){ const rv=percentRectViewport(cfg.steps[k]); if(pointInRect(x,y,rv)) return k; }
  return -1;
}

// --- interactions ---
// Toggle help mode on button
btnToggle.addEventListener('click',(e)=>{ e.stopPropagation(); helpArmed? disarm() : arm(); });

// Global pointer handlers (click + touch)
function onPointer(e){
  // ignore our own UI
  if(e.target.closest('#ha-toggle,#ha-tip')) return;
  if(!helpArmed) return;

  const {x,y}=getClientXY(e);
  const idx=stepAtPoint(x,y);
  if(idx>=0){
    e.preventDefault();
    showStep(idx);     // show the selected area tip
  }else{
    // clicked outside any mapped area: just hide current overlay but stay in help mode
    hideOverlay();
  }
}
document.addEventListener('click', onPointer, true);
document.addEventListener('touchstart', onPointer, {passive:false, capture:true});

// Close actions: only hide overlay (keep helpArmed = true)
btnClose && btnClose.addEventListener('click', (e)=>{ e.stopPropagation(); hideOverlay(); });
mask     && mask.addEventListener('click', (e)=>{ e.stopPropagation(); hideOverlay(); });
document.addEventListener('keydown', (e)=>{ if(e.key==='Escape'){ hideOverlay(); } });

// Keep aligned while open
window.addEventListener('resize', ()=>{ if(currentIdx>=0) showStep(currentIdx); });
window.addEventListener('scroll',  ()=>{ if(currentIdx>=0) showStep(currentIdx); }, {passive:true});

})();
"""
soup.body.append(runtime)

# --- save & download ---
pathlib.Path(OUT_HTML).write_text(str(soup), encoding="utf-8")
print("Saved:", OUT_HTML)
files.download(OUT_HTML)


Saved: hypostat_click_help_toggle_sticky.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ---------------------------------------------------------
# FULL WORKING VERSION (STABLE + FILTER-AWARE)
# Chart.js-aware Chart Q&A with expanded questions
# ---------------------------------------------------------
import pathlib
from bs4 import BeautifulSoup
from google.colab import files

IN_HTML = "hypostat_click_help_toggle_sticky.html"
OUT_HTML = "hypostat_click_help_toggle_sticky_chart_chat.html"

soup = BeautifulSoup(
    pathlib.Path(IN_HTML).read_text(encoding="utf-8"),
    "lxml"
)

# ---------------- CSS ----------------
style = soup.new_tag("style", id="ha-chart-chat-style")
style.string = """
.ha-chart-anchor{position:relative!important;}
.ha-info-btn{
  position:absolute;top:10px;right:10px;z-index:2147483646;
  width:28px;height:28px;border-radius:50%;
  border:1px solid rgba(255,255,255,.2);
  background:#0b1220;color:#fff;cursor:pointer;
}
#ha-chat{
  position:fixed;right:18px;bottom:78px;
  width:400px;height:520px;display:none;flex-direction:column;
  background:#0b1220;color:#e6edf3;z-index:2147483647;
  border-radius:16px;border:1px solid #243042;
  box-shadow:0 25px 80px rgba(0,0,0,.6);
}
#ha-chat.ha-open{display:flex;}
#ha-chat header{
  padding:10px;border-bottom:1px solid #243042;
  display:flex;justify-content:space-between;align-items:center;
}
#ha-chat .close{
  background:none;border:none;color:white;
  font-size:18px;cursor:pointer;
}
#ha-chat .msgs{flex:1;overflow:auto;padding:10px;font-size:13px}
.ha-msg{margin-bottom:8px;padding:8px 10px;border-radius:10px;max-width:92%}
.ha-msg.user{background:#1f2937;margin-left:auto}
.ha-msg.bot{background:#111827}
#ha-chat .composer{
  border-top:1px solid #243042;padding:10px;display:flex;gap:6px
}
#ha-chat textarea{
  flex:1;background:#020617;color:#fff;
  border:1px solid #243042;border-radius:8px;padding:6px
}
"""
soup.head.append(style)

# ---------------- CHAT HTML ----------------
chat_html = BeautifulSoup("""
<div id="ha-chat">
  <header>
    <div>
      <b>Chart Q&A</b><br>
      <span id="ha-chat-subtitle" style="font-size:12px;opacity:.7"></span>
    </div>
    <button class="close" id="ha-chat-close">✕</button>
  </header>
  <div class="msgs" id="ha-chat-msgs"></div>
  <div class="composer">
    <textarea id="ha-chat-input" rows="2"
      placeholder="Ask about this chart..."></textarea>
    <button id="ha-chat-send">➤</button>
  </div>
</div>
""", "lxml")
soup.body.append(chat_html)

# ---------------- RUNTIME JS ----------------
script = soup.new_tag("script")
script.string = r"""
(function(){

const chat=document.getElementById('ha-chat');
const msgs=document.getElementById('ha-chat-msgs');
const input=document.getElementById('ha-chat-input');
const send=document.getElementById('ha-chat-send');
const closeBtn=document.getElementById('ha-chat-close');
const subtitle=document.getElementById('ha-chat-subtitle');

let activeChartIndex=null;
let lastSignature=null;

/* ---------- helpers ---------- */
function add(role,text){
  const d=document.createElement('div');
  d.className='ha-msg '+role;
  d.textContent=text;
  msgs.appendChild(d);
  msgs.scrollTop=msgs.scrollHeight;
}

function chartSignature(chart){
  return JSON.stringify({
    labels: chart.data.labels,
    datasets: chart.data.datasets.map(d=>({
      label:d.label,
      hidden:d.hidden,
      data:d.data
    }))
  });
}

function analyze(chart){
  const d=chart.data;
  const visible=d.datasets.filter(x=>!x.hidden);
  let vals=[];
  visible.forEach(x=>x.data.forEach(v=>typeof v==='number'&&vals.push(v)));

  const sig=chartSignature(chart);
  const changed=lastSignature && lastSignature!==sig;
  lastSignature=sig;

  let trend='stable';
  if(vals.length>1){
    if(vals.at(-1)>vals[0]) trend='upward';
    if(vals.at(-1)<vals[0]) trend='downward';
  }

  return {
    changed,
    years:d.labels||[],
    countries:visible.map(x=>x.label),
    min:vals.length?Math.min(...vals):null,
    max:vals.length?Math.max(...vals):null,
    trend
  };
}

function answer(q){
  const charts=[...Object.values(Chart.instances||{})];
  if(activeChartIndex===null||!charts[activeChartIndex])
    return 'Please select a chart first.';

  const c=analyze(charts[activeChartIndex]);
  q=q.toLowerCase();

  if(q.includes('what')&&q.includes('show')){
    return `This chart shows data from ${c.years[0]}–${c.years.at(-1)} ` +
           `for ${c.countries.length} selected countries: ${c.countries.join(', ')}.`;
  }

  if(q.includes('takeaway')||q.includes('key')){
    return `Key takeaway: an overall ${c.trend} pattern across the selected countries, ` +
           `with values ranging roughly from ${c.min} to ${c.max}.`;
  }

  if(q.includes('trend')){
    return `The dominant trend in the current view is ${c.trend}.`;
  }

  if(q.includes('change')||q.includes('filter')){
    return c.changed
      ? 'The chart has changed due to filters. Countries, values, or trends are different from the previous view.'
      : 'No filter-driven change detected since the last view.';
  }

  if(q.includes('country')){
    return `Currently visible countries: ${c.countries.join(', ')||'none'}.`;
  }

  if(q.includes('range')||q.includes('value')){
    return `Values in the current view range from approximately ${c.min} to ${c.max}.`;
  }

  if(q.includes('outlier')){
    return `Outliers would appear outside the ${c.min}–${c.max} range. Look for isolated spikes or dips.`;
  }

  if(q.includes('compare')){
    return `Comparing first vs last year shows a ${c.trend} movement overall.`;
  }

  return (
    'You can ask:\n' +
    '• What does this chart show?\n' +
    '• What is the key takeaway?\n' +
    '• What trend do you see?\n' +
    '• How did filters change the chart?\n' +
    '• Which countries are selected?\n' +
    '• What is the value range?\n' +
    '• Are there outliers?\n' +
    '• Compare first vs last year'
  );
}

/* ---------- chat flow ---------- */
send.onclick=()=>{
  const q=input.value.trim();
  if(!q) return;
  add('user',q);
  input.value='';
  add('bot',answer(q));
};

closeBtn.onclick=()=>chat.classList.remove('ha-open');

/* ---------- attach info buttons ---------- */
function attach(){
  const charts=[...Object.values(Chart.instances||{})];
  charts.forEach((chart,idx)=>{
    const root=chart.canvas?.parentElement;
    if(!root||root.querySelector('.ha-info-btn')) return;
    root.classList.add('ha-chart-anchor');

    const b=document.createElement('button');
    b.className='ha-info-btn';
    b.textContent='ⓘ';
    b.onclick=e=>{
      e.stopPropagation();
      activeChartIndex=idx;
      subtitle.textContent='Chart selected';
      chat.classList.add('ha-open');
      if(!msgs.children.length){
        add('bot','Ask a question about this chart.');
      }
    };
    root.appendChild(b);
  });
}

/* wait for charts */
const poll=setInterval(()=>{
  if(window.Chart&&Object.keys(Chart.instances||{}).length){
    attach();
  }
},500);

new MutationObserver(attach)
  .observe(document.body,{childList:true,subtree:true});

})();
"""
soup.body.append(script)

# ---------------- SAVE ----------------
pathlib.Path(OUT_HTML).write_text(str(soup), encoding="utf-8")
print("Saved:", OUT_HTML)
files.download(OUT_HTML)



Saved: hypostat_click_help_toggle_sticky_chart_chat.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>